# AI Engineering: Text Processing

## >Imports

In [43]:
!pip install tf keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 3.8 MB/s eta 0:00:00


In [47]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow.keras as keras
from numpy.linalg import norm
import tf_keras
import tensorflow_datasets as tfds

## >Text Embedding

In [5]:
MODEL_URL = 'https://tfhub.dev/google/universal-sentence-encoder/4'

In [6]:
model = hub.load(MODEL_URL)

In [7]:
def embed(input_text, embed_model=model):
    return embed_model(input_text)

In [8]:
embeddings = embed(['This is a sentence.'])

In [13]:
embeddings[0].numpy()[0:10]

array([ 0.02881767, -0.02020015,  0.01069628,  0.0385053 , -0.09253702,
        0.01752774, -0.04711751,  0.0478521 ,  0.01430713,  0.02635952],
      dtype=float32)

## Semantic Similarity *Scoring*

In [38]:
def cos_sim(A,B):
    return np.inner(A,B)/(norm(A)*norm(B))

def euclidiean(A,B):
    return norm(A-B)

In [39]:
def is_it_sim(textA, textB, thresh=.2, sim_func=cos_sim):
    embeddingA = embed([textA])
    embeddingB = embed([textB])
    sim_score = sim_func(embeddingA,embeddingB)
    if sim_score>thresh:
        print('They are SIMILAR')
    else:
        print('They are NOT similar')
    return sim_score, sim_score > thresh

In [40]:
questionA = 'This is a technology Company that builds computers'
answerA = 'I have a deslicious fruit called an apple'
answerB = 'I am writing a program on my Apple desktop'
answerC = 'My favorite PC is not an HP'

In [42]:
is_it_sim(questionA, answerA)
is_it_sim(questionA, answerB)
is_it_sim(questionA, answerC)
is_it_sim(answerB, answerC)

They are NOT similar
They are SIMILAR
They are SIMILAR
They are SIMILAR


(array([[0.3224339]], dtype=float32), array([[ True]]))

## >Text Classification using Embedding & GUSE

In [48]:
train_data, validation_data, test_data = tfds.load(
    name="imdb_reviews",
    split=('train[:60%]','train[60%:]','test'),
    as_supervised =True)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.2LQWOR_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.2LQWOR_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.2LQWOR_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


In [49]:
train_examples_batch, train_labels_batch = next(iter(train_data.batch(10)))

In [51]:
train_examples_batch[0]

<tf.Tensor: shape=(), dtype=string, numpy=b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.">

## >Load Hub Data and Create a Hub Layer

In [82]:
embedding_model_url='https://tfhub.dev/google/universal-sentence-encoder/4'

In [83]:
hub_layer = hub.KerasLayer(embedding_model_url,
                           input_shape=[],
                           dtype=tf.string,
                           trainable=False)

## >Build a NN Text Classifier

In [84]:
nlp_model = tf_keras.Sequential()

In [85]:
nlp_model.add(hub_layer)

nlp_model.add(tf_keras.layers.Dense(256, activation='relu'))
nlp_model.add(tf_keras.layers.Dropout(0.1))

nlp_model.add(tf_keras.layers.Dense(128, activation='relu'))
nlp_model.add(tf_keras.layers.Dropout(0.1))

nlp_model.add(tf_keras.layers.Dense(64, activation='relu'))

nlp_model.add(tf_keras.layers.Dense(1, activation='sigmoid'))


In [86]:
nlp_model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer_3 (KerasLayer)  (None, 512)               256797824 
                                                                 
 dense_7 (Dense)             (None, 256)               131328    
                                                                 
 dropout_4 (Dropout)         (None, 256)               0         
                                                                 
 dense_8 (Dense)             (None, 128)               32896     
                                                                 
 dropout_5 (Dropout)         (None, 128)               0         
                                                                 
 dense_9 (Dense)             (None, 64)                8256      
                                                                 
 dense_10 (Dense)            (None, 1)                

## >Compile and Train NLP Model

In [87]:
nlp_model.compile(optimizer='Adam',
                  loss=tf_keras.losses.BinaryCrossentropy(from_logits=False),
                  metrics=['binary_accuracy'])

In [91]:
history = nlp_model.fit(train_data.batch(512),
                    epochs=20,
                    validation_data=validation_data.batch(512),
                    verbose=1)

Epoch 1/20
30/30 [==============================] - 55s 2s/step - loss: 0.5023 - binary_accuracy: 0.7922 - val_loss: 0.3584 - val_binary_accuracy: 0.8425
Epoch 2/20
30/30 [==============================] - 49s 2s/step - loss: 0.3422 - binary_accuracy: 0.8507 - val_loss: 0.3307 - val_binary_accuracy: 0.8557
Epoch 3/20
30/30 [==============================] - 49s 2s/step - loss: 0.3212 - binary_accuracy: 0.8622 - val_loss: 0.3291 - val_binary_accuracy: 0.8570
Epoch 4/20
30/30 [==============================] - 49s 2s/step - loss: 0.3127 - binary_accuracy: 0.8675 - val_loss: 0.3269 - val_binary_accuracy: 0.8578
Epoch 5/20
30/30 [==============================] - 48s 2s/step - loss: 0.3018 - binary_accuracy: 0.8733 - val_loss: 0.3245 - val_binary_accuracy: 0.8600
Epoch 6/20
 3/30 [==>...........................] - ETA: 23s - loss: 0.3027 - binary_accuracy: 0.8724

KeyboardInterrupt: 

## >Model Evaluation

In [92]:
results = nlp_model.evaluate(test_data.batch(512), verbose=1)

49/49 [==============================] - 46s 932ms/step - loss: 0.3322 - binary_accuracy: 0.8555


## >Save Model Weights

In [94]:
nlp_model.save('/content/NLP_Model')

## >Inference Function

In [126]:
def get_sentiment(text, model=nlp_model, thresh=0.5):
  p_hat =model.predict([text])[0][0]
  out = (p_hat>thresh).astype('int32')

  print("Viewer Comment:\n"+text+"\n\nThe Review was:")
  if out: print("->It was Good!")
  else: print("->Bad Movie!")
  return int(p_hat*100)/100


In [133]:
get_sentiment('Can\'t imagine a better movie. \n  All other movies wake up in a cold sweat having been haunted by the spectre of the movie they could never be.\n Will watch again and again until my flesh has melded to the cinema\'s seats and they have to peel me off like an overcooked steak from a skillet.')

1/1 [==============================] - 0s 38ms/step
Viewer Comment:
Can't imagine a better movie. 
  All other movies wake up in a cold sweat having been haunted by the spectre of the movie they could never be.
 Will watch again and again until my flesh has melded to the cinema's seats and they have to peel me off like an overcooked steak from a skillet.

The Review was:
->Bad Movie!


0.03